### Dataset attribute descriptions

**CustomerID**: A unique ID that identifies each customer.

**Count**: A value used in reporting/dashboarding to sum up the number of customers in a filtered set.

**Country**: The country of the customer’s primary residence.

**State**: The state of the customer’s primary residence.

**City**: The city of the customer’s primary residence.

**Zip Code**: The zip code of the customer’s primary residence.

**Lat Long**: The combined latitude and longitude of the customer’s primary residence.

**Latitude**: The latitude of the customer’s primary residence.

**Longitude**: The longitude of the customer’s primary residence.

**Gender**: The customer’s gender: Male, Female

**Senior Citizen**: Indicates if the customer is 65 or older: Yes, No

**Partner**: Indicate if the customer has a partner: Yes, No

**Dependents**: Indicates if the customer lives with any dependents: Yes, No. Dependents could be children, parents, grandparents, etc.

**Tenure Months**: Indicates the total amount of months that the customer has been with the company by the end of the quarter specified above.

**Phone Service**: Indicates if the customer subscribes to home phone service with the company: Yes, No

**Multiple Lines**: Indicates if the customer subscribes to multiple telephone lines with the company: Yes, No

**Internet Service**: Indicates if the customer subscribes to Internet service with the company: No, DSL, Fiber Optic, Cable.

**Online Security**: Indicates if the customer subscribes to an additional online security service provided by the company: Yes, No

**Online Backup**: Indicates if the customer subscribes to an additional online backup service provided by the company: Yes, No

**Device Protection**: Indicates if the customer subscribes to an additional device protection plan for their Internet equipment provided by the company: Yes, No

**Tech Support**: Indicates if the customer subscribes to an additional technical support plan from the company with reduced wait times: Yes, No

**Streaming TV**: Indicates if the customer uses their Internet service to stream television programing from a third party provider: Yes, No. The company does not charge an additional fee for this service.

**Streaming Movies**: Indicates if the customer uses their Internet service to stream movies from a third party provider: Yes, No. The company does not charge an additional fee for this service.

**Contract**: Indicates the customer’s current contract type: Month-to-Month, One Year, Two Year.

**Paperless Billing**: Indicates if the customer has chosen paperless billing: Yes, No

**Payment Method**: Indicates how the customer pays their bill: Bank Withdrawal, Credit Card, Mailed Check

**Monthly Charge**: Indicates the customer’s current total monthly charge for all their services from the company.

**Total Charges**: Indicates the customer’s total charges, calculated to the end of the quarter specified above.

**Churn Label**: Yes = the customer left the company this quarter. No = the customer remained with the company. Directly related to Churn Value.

**Churn Value**: 1 = the customer left the company this quarter. 0 = the customer remained with the company. Directly related to Churn Label.

**Churn Score**: A value from 0-100 that is calculated using the predictive tool IBM SPSS Modeler. The model incorporates multiple factors known to cause churn. The higher the score, the more likely the customer will churn.

**CLTV**: Customer Lifetime Value. A predicted CLTV is calculated using corporate formulas and existing data. The higher the value, the more valuable the customer. High value customers should be monitored for churn.

**Churn Reason**: A customer’s specific reason for leaving the company. Directly related to Churn Category.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_excel("../data/Telco_Customer_Churn.xlsx")

In [ ]:
print(f"Number of columns: {df.shape[1]}\n") # amount of cols
df.info() # amount of rows, columns and data types
df.describe() # statistical summary of numerical columns

In [ ]:
# Lower each column name
df.columns = df.columns.str.lower()

In [ ]:
# Gender distribution
"""
male = df[df["Gender"]=="Male"]
female = df[df["Gender"]=="Female"]

male_count = male.shape[0]
female_count = female.shape[0]
total_count = df.shape[0]

male_churn_count = male[male["Churn Value"] == 1].shape[0]
female_churn_count = female[female["Churn Value"] == 1].shape[0]

churn_by_gender = pd.DataFrame({
    "Gender" : ["Male", "Male", "Female", "Female"],
    "Churn Status" : ["Churned", "Not Churned", "Churned", "Not Churned"],
    "Count" : [male_churn_count, male_count - male_churn_count, female_churn_count, female_count - female_churn_count]
})
"""

gender_summary = pd.crosstab(
    df["gender"],
    df["churn value"].map({1: "churned", 0: "not churned"}),
    margins=True, # automatically calculates total
    margins_name="total"
)

print(gender_summary)

In [ ]:
# Contract Type
churn_by_contract = pd.crosstab(
    df["contract"], 
    df["churn value"].map({1 : "churned", 0 : "not churned"}), 
    normalize=True, 
    margins = True, 
    margins_name="total"
    )

print(churn_by_contract)

In [ ]:
# Distribution of Churn Reason
churn_with_reason = df["churn reason"].value_counts()
#print(churn_with_reason)
                                                  
# Churn Without Reason Provided
churn_without_reason = df[(df["churn reason"].isna()) & (df["churn value"] == 1)].shape[0]      

print(f"\nChurn with a reason given: {churn_with_reason.sum()}")
print(f"Churn without reason: {churn_without_reason}")                                    


def churn_reason_count(churn_reason: list[str]) -> int:
    return sum(churn_with_reason[reason] for reason in churn_reason)


churn_reasons: dict[str, list[str]] = {
    "Competitor" : ["Competitor offered higher download speeds", "Competitor offered more data", "Competitor made better offer", "Competitor had better devices"],
    "Service" : ["Network reliability", "Limited range of services"],
    "Pricing" : ["Price too high", "Extra data charges", "Long distance charges", "Lack of affordable download/upload speed"],
    "Customer Support" : ["Attitude of support person", "Attitude of service provider", "Lack of self-service on Website", "Poor expertise of phone support", "Poor expertise of online support"],
    "Personal" : ["Service dissatisfaction", "Moved", "Deceased"],
    "Miscellaneous" : ["Don't know"]
}

churn_category_count: dict[str, int] = {
    category : churn_reason_count(reasons) for category, reasons in churn_reasons.items()
}

for reason, count in churn_category_count.items():
    print(f"Churn because of {reason} reason: {count}")

In [ ]:
labels = list(churn_category_count.keys())
data = list(churn_category_count.values())

plt.figure(figsize=(10,6))
colors = sns.color_palette("dark")

plt.pie(data, labels=labels, colors=colors, autopct="%.2f%%", startangle=90) # start at north
plt.title("Reason for customer churn")

plt.show()

In [ ]:
location_count = df["city"].value_counts().reset_index(name="counts")
churn_by_location = df[df["churn value"] == 1].groupby("city")["churn value"].sum().reset_index(name="churn count")

location_summary = location_count.merge(churn_by_location, on = "city")
location_summary["churn_percentage"] = round(100 * location_summary["churn count"]/location_summary["counts"], 2)

#location_summary = location_summary.sort_values(by="churn_percentage", ascending=False)
#print(location_summary.to_string(index=False))

# Churn because of competitor reasons, split by zip code

competitor_churn_by_location = df[(df["churn value"] == 1) & (df["churn reason"].isin(churn_reasons["Competitor"]))].groupby("zip code")["churn value"].sum().reset_index(name="churn count due to competitor").sort_values(by="churn count due to competitor", ascending=False)
print(competitor_churn_by_location.to_string(index=False))